# Balanced T1 Test Evaluation

Run this notebook only after the class-balanced adapter has passed the validation gate. It evaluates the frozen adapter once on the held-out test split and does not retrain. The compact test bundle stores these held-out records in `val.jsonl`; their metadata and image paths still identify the test split.

In [ ]:
%pip install -q -U "transformers>=4.50.0" peft accelerate bitsandbytes pillow scikit-learn safetensors
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import tarfile
import zipfile

print("Upload colab-t1-test-bundle.tar.gz and the balanced adapter ZIP.")
uploaded = files.upload()
bundle_name = next(name for name in uploaded if name.endswith(".tar.gz") and "test" in name)
adapter_name = next(name for name in uploaded if name.endswith(".zip") and "balanced" in name)
root = Path("/content/balanced_test_bundle")
root.mkdir(exist_ok=True)
with tarfile.open(Path("/content") / bundle_name, "r:gz") as archive:
    archive.extractall(root)
with zipfile.ZipFile(Path("/content") / adapter_name) as archive:
    archive.extractall(root)
adapter_configs = list(root.rglob("adapter_config.json"))
adapter_dir = next(path.parent for path in adapter_configs if "balanced" in str(path.parent))
test_path = root / "data/secondary/instruction_tuning_mixed/val.jsonl"
assert test_path.exists(), test_path
assert sum(1 for line in test_path.read_text().splitlines() if line.strip()) == 477
print("Bundle root:", root)
print("Balanced adapter:", adapter_dir)
print("Test records:", 477)

In [ ]:
import json
import subprocess
import sys

runner = root / "scripts/train_t1_qlora.py"
command = [
    sys.executable, "-u", str(runner),
    "--root", str(root),
    "--output", str(adapter_dir),
    "--eval-only",
    "--split", "val",
    "--max-val", "477",
    "--max-new-tokens", "8",
]
subprocess.run(command, check=True)
metrics = json.loads((adapter_dir / "t1_metrics.json").read_text())
print(json.dumps(metrics, indent=2))
assert metrics["records"] == 477
assert metrics["invalid_outputs"] == 0

In [ ]:
from google.colab import files

results_zip = shutil.make_archive("/content/t1-balanced-test-results", "zip", root_dir=adapter_dir.parent, base_dir=adapter_dir.name)
print("Download:", results_zip)
files.download(results_zip)